In [ ]:
import logging

import numpy as np
import openeo.processes
import shapely

import openeo

from utils import utils

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
spatial_extent = {
    "west": 30.5503711040000994,
    "south": 1.0709279050000799,
    "east": 31.2229521229999989,
    "north": 1.5469373050000299,
}
temporal_extent = ["2019-10-01", "2025-04-01"]  # pad by ~3 months
band = "VH"
bands = [band]
instrument_mode = "IW"
orbit_state = "ascending"
relative_orbit = 101

speckle_filter_radius = 2
speckle_filter_cv_noise = 1 / np.sqrt(4)
speckle_filter_window_size = 5

resample_spatial_resolution = 30  # m

logistic_window_size = 11
logistic_steepness_parameter = -2.0  # dimensionless

temporal_variability_threshold = 0.5  # units: dB
flattening_threshold = 0.12  # units: dimensionless

logistic_sse_percentile = 0.95  # p95 is correct

min_connected_area = 10000  # m^2

In [ ]:
# very small test AOI
x = 30.944
y = 1.273
delta = 0.1
spatial_extent = {
    "west": x,
    "south": y,
    "east": x + delta,
    "north": y + delta,
}

In [ ]:
# spatial extent as dict of Polygon geometry
spatial_extent = shapely.geometry.mapping(
    shapely.box(
        xmin=spatial_extent["west"],
        ymin=spatial_extent["south"],
        xmax=spatial_extent["east"],
        ymax=spatial_extent["north"],
    )
)

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

## Sentinel 1

In [ ]:
# load results from previous batch job
JOB_ID = "j-260713124951418ebf4a620f7e421671"

s1_dB = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0000_s1_dB",
        },
    )
)

In [ ]:
# load_stac adds a time dimension 😠
s1_dB = s1_dB.drop_dimension("t")

In [ ]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0001_s1_dB",
        },
    )
)

# load forest baseline

In [ ]:
# load results from previous batch job
JOB_ID = "j-26071711012842baa6a6029499ef8ade"

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0010_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_forest_baseline_mask",
        },
    )
)

In [ ]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

In [ ]:
# load_stac incorrectly sets nodata=0 😠
forest_baseline_mask = forest_baseline_mask == 1

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0012_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0012_forest_baseline_mask",
        },
    )
)

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
forest_baseline_mask = forest_baseline_mask.resample_cube_spatial(
    s1_dB,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0015_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0015_forest_baseline_mask",
        },
    )
)

## temporal variability mask

In [ ]:
# mask of interesting pixels

# band math
temporal_variability_mask = s1_dB.band("sd") >= temporal_variability_threshold

In [ ]:
process_graph_results.append(
    temporal_variability_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0020_temporal_variability_mask",
        },
    )
)
process_graph_results.append(
    temporal_variability_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_temporal_variability_mask",
        },
    )
)

# flattening mask

In [ ]:
# TODO: ATBD says denominator is abs(gamma_max)
# but code uses p05
# Dascalu 2023 has abs(gamma_max)


def flattening_reducer(
    bands: openeo.processes.ProcessBuilder,
) -> openeo.processes.ProcessBuilder:
    p05 = bands.array_element(label="p05")
    p95 = bands.array_element(label="p95")
    return (p95 - p05) / p95.absolute()


flattening = s1_dB.reduce_bands(flattening_reducer)

In [ ]:
assert flattening._in_bandmath_mode()

In [ ]:
process_graph_results.append(
    flattening.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0030_flattening",
        },
    )
)
process_graph_results.append(
    flattening.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_flattening",
        },
    )
)

In [ ]:
# mask of good pixels where flattening >= flattening_threshold

# band math
flattening_mask = flattening >= flattening_threshold

In [ ]:
process_graph_results.append(
    flattening_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0040_flattening_mask",
        },
    )
)
process_graph_results.append(
    flattening_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_flattening_mask",
        },
    )
)

# candidate mask

In [ ]:
# forest_baseline_mask.band("B0") & temporal_variability_mask & flattening_mask
# BandMathException: 'Band math' between bands of different data cubes is not supported yet. 🙁

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

b1 = forest_baseline_mask.rename_labels(
    dimension="bands", target=["forest_baseline_mask"]
)
b2 = temporal_variability_mask.add_dimension(
    "bands", label="temporal_variability_mask", type="bands"
)
b3 = flattening_mask.add_dimension("bands", label="flattening_mask", type="bands")

combined = b1.merge_cubes(b2).merge_cubes(b3)

In [ ]:
process_graph_results.append(
    combined.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0050_combined",
        },
    )
)
process_graph_results.append(
    combined.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_combined",
        },
    )
)
# check dtype of the merge_cubes bands - uint8
process_graph_results.append(
    combined.band("temporal_variability_mask").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_combined_temporal_variability_mask",
        },
    )
)
process_graph_results.append(
    combined.band("flattening_mask").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_combined_flattening_mask",
        },
    )
)
process_graph_results.append(
    combined.band("forest_baseline_mask").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_combined_forest_baseline_mask",
        },
    )
)

In [ ]:
# candidate pixel = 1
candidate_mask = (
    combined.band("temporal_variability_mask")
    & combined.band("flattening_mask")
    & combined.band("forest_baseline_mask")
)

In [ ]:
process_graph_results.append(
    candidate_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0051_candidate_mask",
        },
    )
)
process_graph_results.append(
    candidate_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0051_candidate_mask",
        },
    )
)

In [ ]:
inverse_candidate_mask = utils.logical_not(candidate_mask)

In [ ]:
process_graph_results.append(
    inverse_candidate_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0052_inverse_candidate_mask",
        },
    )
)
process_graph_results.append(
    inverse_candidate_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0052_inverse_candidate_mask",
        },
    )
)

## Mask based on percentile of min_sse

Here we discard candidate deforestation events where the SSE (goodness of fit) is above a certain percentile.

This is the final mask of deforestation event detections

In [ ]:
# only consider candidate pixels
min_sse_canditates = s1_dB.filter_bands("min_sse").mask(inverse_candidate_mask)

In [ ]:
assert not min_sse_canditates._in_bandmath_mode()

In [ ]:
process_graph_results.append(
    min_sse_canditates.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0060_min_sse_canditates",
        },
    )
)
process_graph_results.append(
    min_sse_canditates.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_min_sse_canditates",
        },
    )
)

In [ ]:
measured_percentile = utils.percentile_cube(
    cube=min_sse_canditates,
    aoi=spatial_extent,
    percentile=logistic_sse_percentile,
)

In [ ]:
process_graph_results.append(
    measured_percentile.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0070_measured_percentile",
        },
    )
)
process_graph_results.append(
    measured_percentile.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_measured_percentile",
        },
    )
)

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

b1 = min_sse_canditates.rename_labels(dimension="bands", target=["min_sse_canditates"])
b2 = measured_percentile.rename_labels(
    dimension="bands", target=["measured_percentile"]
)

combined = b1.merge_cubes(b2)

In [ ]:
# mask of good pixels where min_sse <= logistic_sse_percentile

# band math
deforestation_event_mask = combined.band("min_sse_canditates") <= combined.band(
    "measured_percentile"
)

In [ ]:
process_graph_results.append(
    deforestation_event_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0080_deforestation_event_mask",
        },
    )
)
process_graph_results.append(
    deforestation_event_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0080_deforestation_event_mask",
        },
    )
)

In [ ]:
inverse_deforestation_event_mask = utils.logical_not(deforestation_event_mask)

In [ ]:
process_graph_results.append(
    inverse_deforestation_event_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0081_inverse_deforestation_event_mask",
        },
    )
)
process_graph_results.append(
    inverse_deforestation_event_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0081_inverse_deforestation_event_mask",
        },
    )
)

# Apply deforestation event mask to `min_sse_t`

In [ ]:
min_sse_t_masked = s1_dB.band("min_sse_t").mask(inverse_deforestation_event_mask)

In [ ]:
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0090_min_sse_t_masked",
        },
    )
)
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0090_min_sse_t_masked",
        },
    )
)

## mask based on connectivity

of the natural forest remaining, are regions of forest too small to meet the minimum connected area threshold?

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

b1 = forest_baseline_mask.rename_labels(
    dimension="bands", target=["forest_baseline_mask"]
)
b2 = inverse_deforestation_event_mask.add_dimension(
    "bands", label="inverse_deforestation_event_mask", type="bands"
)

combined = b1.merge_cubes(b2)

In [ ]:
process_graph_results.append(
    combined.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0100_combined",
        },
    )
)
process_graph_results.append(
    combined.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0100_combined",
        },
    )
)
# check dtype of the merge_cubes bands - uint8
process_graph_results.append(
    combined.band("forest_baseline_mask").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0100_combined_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    combined.band("inverse_deforestation_event_mask").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0100_combined_inverse_deforestation_event_mask",
        },
    )
)

In [ ]:
remaining_forest_mask = combined.band("forest_baseline_mask") & combined.band(
    "inverse_deforestation_event_mask"
)

In [ ]:
process_graph_results.append(
    remaining_forest_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0110_remaining_forest_mask",
        },
    )
)
process_graph_results.append(
    remaining_forest_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0110_remaining_forest_mask",
        },
    )
)

In [ ]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        "pixel_area": resample_spatial_resolution * resample_spatial_resolution,
        "min_connected_area": min_connected_area,
    },
)

In [ ]:
# mask where 1 = small region to be excluded
small_region_mask = remaining_forest_mask.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0120_small_region_mask",
        },
    )
)

In [ ]:
# apply_neighborhood UDF seems to return float32, even if it's a mask
# data types: https://github.com/locationtech/geotrellis/blob/master/raster/src/main/scala/geotrellis/raster/CellType.scala
small_region_mask = small_region_mask.convert_data_type("bool")

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0121_small_region_mask",
        },
    )
)

In [ ]:
# stack min_sse_t_masked + small_region_mask into a single datacube
# in preparation for the nearest neighbour fill UDF

_data = min_sse_t_masked.add_dimension("bands", label="data", type="bands")
_mask = small_region_mask.add_dimension("bands", label="mask", type="bands")
combined = _data.merge_cubes(_mask)

In [ ]:
process_graph_results.append(
    combined.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0130_combined",
        },
    )
)
process_graph_results.append(
    combined.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0130_combined",
        },
    )
)

In [ ]:
nearest_neighbour_fill_udf = openeo.UDF.from_file(
    "../udf/nearest_neighbour_fill.py",
    runtime="Python",
    version="3.11",
)

In [ ]:
min_sse_t = combined.apply_neighborhood(
    nearest_neighbour_fill_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    min_sse_t.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0140_min_sse_t",
        },
    )
)
process_graph_results.append(
    min_sse_t.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0140_min_sse_t",
        },
    )
)

# Run batch job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)

In [ ]:
job = multi_result.create_job()
job.start_and_wait()
# Inspect job.logs() if it fails

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)